In [1]:
"""
Descriptive Statistics Pipeline (Thesis / GitHub-ready)

What this script does (end-to-end):
1) Loads similarity matrices for two years (e.g., 2019 and 2024)
2) Computes pairwise (unique) similarity distributions (upper triangle only)
3) Computes year-to-year changes per firm-pair (Δ = 2024 − 2019)
4) Produces key descriptive tables:
   - Pairwise descriptives per year (mean, median, sd, quartiles, min, max)
   - Change descriptives (Δ)
   - Convergence/divergence counts and percentages
   - Top converging pairs and top diverging pairs
5) Loads firm-level (same-firm) 2019→2024 results (optional) and summarizes:
   - Firm-level similarity descriptives
   - Text-length change summaries (words, Δ words, Δ%)
6) Exports everything to a single Excel workbook 

Important notes for reproducibility:
- This script does NOT require raw PDFs or extracted text.
- It only needs the similarity matrices  -> this similarity matrixes have been made by myself in excel and uploaded to outputs / similarity_excel / fe. "similarity_matrix_2019.xlsx"
- It avoids causal inference (purely descriptive-comparative).
"""

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ==========================================================
# 1) CONFIGURATION (EDIT THESE PATHS)
# ==========================================================

BASE = Path(".").resolve()

# --- Required: similarity matrices (Excel or CSV) ---
# Expected format:
# - Square matrix (same firms as rows and columns)
# - Row/column labels are firm identifiers
# - Diagonal can be 1.0 (self-similarity), will be ignored
MATRIX_2019_PATH = BASE / "outputs" / "similarity_excel" / "similarity_matrix_2019.xlsx"
MATRIX_2024_PATH = BASE / "outputs" / "similarity_excel" / "similarity_matrix_2024.xlsx"

# --- Optional: firm-level same-firm 2019→2024 results ---
# Expected columns (from your export script):
# firm, year_a, year_b, cosine_similarity, words_year_a, words_year_b, delta_words, delta_words_pct, ...
FIRM_LEVEL_PATH = BASE / "outputs" / "firm_year_2019_2024_main.csv"  # set to None if not used

# --- Output folder ---
OUT_DIR = BASE / "outputs" / "descriptives"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_EXCEL = OUT_DIR / "descriptive_statistics_export.xlsx"

# --- Figures (optional) ---
MAKE_FIGURES = True
FIG_DIR = OUT_DIR / "figures"
if MAKE_FIGURES:
    FIG_DIR.mkdir(parents=True, exist_ok=True)


# ==========================================================
# 2) HELPERS
# ==========================================================

def load_matrix(path: Path) -> pd.DataFrame:
    """Load a similarity matrix from .xlsx or .csv."""
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")

    if path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path, index_col=0)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, index_col=0)
    else:
        raise ValueError("Matrix file must be .xlsx/.xls or .csv")

    # Ensure numeric
    df = df.apply(pd.to_numeric, errors="coerce")
    return df


def ensure_same_order(M_a: pd.DataFrame, M_b: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Ensure both matrices have the same row/column order and firm set.
    This is critical before computing pairwise deltas.
    """
    firms_a = list(M_a.index)
    firms_b = list(M_b.index)

    if set(firms_a) != set(firms_b):
        missing_in_b = sorted(set(firms_a) - set(firms_b))
        missing_in_a = sorted(set(firms_b) - set(firms_a))
        raise ValueError(
            "Firm sets differ between matrices.\n"
            f"Missing in 2024: {missing_in_b}\n"
            f"Missing in 2019: {missing_in_a}"
        )

    firms = sorted(set(firms_a))
    M_a = M_a.loc[firms, firms]
    M_b = M_b.loc[firms, firms]
    return M_a, M_b


def upper_triangle_values(mat: pd.DataFrame) -> np.ndarray:
    """Return unique pairwise values (upper triangle, excluding diagonal)."""
    A = mat.values.astype(float)
    n = A.shape[0]
    vals = A[np.triu_indices(n, k=1)]
    vals = vals[~np.isnan(vals)]
    return vals


def pair_long_format(mat: pd.DataFrame, year: int) -> pd.DataFrame:
    """
    Convert a square similarity matrix into a long dataframe with unique pairs.
    Output columns: year, firm_i, firm_j, similarity
    """
    firms = list(mat.index)
    A = mat.values.astype(float)
    n = A.shape[0]

    rows = []
    iu = np.triu_indices(n, k=1)
    for i, j in zip(iu[0], iu[1]):
        val = A[i, j]
        if np.isnan(val):
            continue
        rows.append({
            "year": year,
            "firm_i": firms[i],
            "firm_j": firms[j],
            "similarity": float(val)
        })
    return pd.DataFrame(rows)


def describe_series(x: np.ndarray | pd.Series) -> dict:
    """Compute descriptive statistics commonly reported in results sections."""
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    return {
        "n_pairs": int(len(x)),
        "mean": float(np.mean(x)) if len(x) else np.nan,
        "median": float(np.median(x)) if len(x) else np.nan,
        "std": float(np.std(x, ddof=1)) if len(x) > 1 else np.nan,
        "min": float(np.min(x)) if len(x) else np.nan,
        "p25": float(np.percentile(x, 25)) if len(x) else np.nan,
        "p75": float(np.percentile(x, 75)) if len(x) else np.nan,
        "max": float(np.max(x)) if len(x) else np.nan,
    }


def safe_round_df(df: pd.DataFrame, decimals: int = 4) -> pd.DataFrame:
    """Round numeric columns for reporting, leaving text columns intact."""
    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_numeric_dtype(out[c]):
            out[c] = out[c].round(decimals)
    return out


# ==========================================================
# 3) LOAD MATRICES + VALIDATE
# ==========================================================

M19 = load_matrix(MATRIX_2019_PATH)
M24 = load_matrix(MATRIX_2024_PATH)

# Align firm sets + order
M19, M24 = ensure_same_order(M19, M24)

# Long format (unique pairs) for each year
pairs19 = pair_long_format(M19, year=2019)
pairs24 = pair_long_format(M24, year=2024)

# Merge pairs to compute deltas per firm-pair
pairs = pairs19.merge(
    pairs24,
    on=["firm_i", "firm_j"],
    suffixes=("_2019", "_2024"),
    how="inner"
)

pairs["delta_similarity"] = pairs["similarity_2024"] - pairs["similarity_2019"]
pairs["direction"] = np.where(pairs["delta_similarity"] >= 0, "convergence (Δ≥0)", "divergence (Δ<0)")


# ==========================================================
# 4) CORE DESCRIPTIVE TABLES (PAIRWISE)
# ==========================================================

vals_2019 = pairs["similarity_2019"].to_numpy()
vals_2024 = pairs["similarity_2024"].to_numpy()
vals_delta = pairs["delta_similarity"].to_numpy()

desc_pairwise = pd.DataFrame({
    "2019": describe_series(vals_2019),
    "2024": describe_series(vals_2024),
    "Δ (2024−2019)": describe_series(vals_delta),
})

# Convergence/divergence counts
direction_counts = (
    pairs["direction"]
    .value_counts()
    .rename_axis("direction")
    .reset_index(name="count")
)
direction_counts["pct"] = direction_counts["count"] / direction_counts["count"].sum()

# Top converging/diverging pairs
top_converging = pairs.sort_values("delta_similarity", ascending=False).head(10).copy()
top_diverging = pairs.sort_values("delta_similarity", ascending=True).head(10).copy()

# Add helpful absolute size
pairs["abs_delta"] = pairs["delta_similarity"].abs()
top_largest_changes = pairs.sort_values("abs_delta", ascending=False).head(10).copy()


# ==========================================================
# 5) OPTIONAL: FIRM-LEVEL (SAME-FIRM) DESCRIPTIVES
# ==========================================================

firm_df = None
firm_desc = None
firm_ranked = None

if FIRM_LEVEL_PATH is not None and Path(FIRM_LEVEL_PATH).exists():
    firm_df = pd.read_csv(FIRM_LEVEL_PATH)

    # Basic checks
    required_cols = {"firm", "cosine_similarity"}
    missing_cols = required_cols - set(firm_df.columns)
    if missing_cols:
        raise ValueError(f"Firm-level file is missing columns: {sorted(missing_cols)}")

    firm_desc = pd.DataFrame({
        "firm_level": describe_series(firm_df["cosine_similarity"].to_numpy())
    })

    # Rank for interpretation
    firm_ranked = firm_df.sort_values("cosine_similarity", ascending=False).reset_index(drop=True)

    # If word columns exist, provide additional summary
    if {"words_year_a", "words_year_b", "delta_words_pct"}.issubset(set(firm_df.columns)):
        # keep as-is; these are already cleaned-token counts (consistent with your NLP pipeline)
        pass


# ==========================================================
# 6) EXPORT TO EXCEL (ONE CLEAN WORKBOOK)
# ==========================================================

with pd.ExcelWriter(OUT_EXCEL, engine="openpyxl") as writer:
    safe_round_df(desc_pairwise).to_excel(writer, sheet_name="pairwise_descriptives")
    safe_round_df(direction_counts, 4).to_excel(writer, sheet_name="convergence_counts", index=False)

    # Pair-level tables for interpretation
    safe_round_df(pairs, 6).to_excel(writer, sheet_name="all_pairs_long", index=False)
    safe_round_df(top_converging, 6).to_excel(writer, sheet_name="top_converging", index=False)
    safe_round_df(top_diverging, 6).to_excel(writer, sheet_name="top_diverging", index=False)
    safe_round_df(top_largest_changes, 6).to_excel(writer, sheet_name="largest_abs_changes", index=False)

    # Optional firm-level sheets
    if firm_df is not None:
        safe_round_df(firm_desc).to_excel(writer, sheet_name="firm_level_descriptives")
        safe_round_df(firm_ranked, 6).to_excel(writer, sheet_name="firm_level_ranked", index=False)

print(f"✅ Descriptive statistics exported to:\n{OUT_EXCEL.resolve()}")



Matplotlib is building the font cache; this may take a moment.


✅ Descriptive statistics exported to:
/Users/pnlmf036/Documents/Thesis_Textanalysis/outputs/descriptives/descriptive_statistics_export.xlsx


/var/folders/jp/slvlq_k14j3_2491rk9fkrdm0000gn/T/ipykernel_26338/3069187562.py:284: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot([vals_2019, vals_2024], labels=["2019", "2024"])


✅ Figures saved to:
/Users/pnlmf036/Documents/Thesis_Textanalysis/outputs/descriptives/figures


In [3]:
"""
Firm-Level MD&A Similarity Analysis (2019–2024)

Purpose
-------
This script computes within-firm linguistic similarity between MD&A disclosures
from two reporting years (e.g., 2019 and 2024). It evaluates how stable each
firm’s narrative reporting remains over time.

Method Overview
---------------
For each firm:
1. Loads cleaned MD&A text files from both years.
2. Computes TF-IDF vector representations.
3. Calculates cosine similarity between the two documents.
4. Computes word counts and length changes.
5. Generates descriptive statistics and ranked summaries.

Outputs
-------
The script exports:
- A main firm-level dataset (CSV + Excel)
- Descriptive statistics (similarity and text length)
- Similarity buckets (high/medium/low stability)
- Top/bottom similarity firms
- Largest text length changes
- Flags for qualitative follow-up
"""

from pathlib import Path
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================================
# 1) USER SETTINGS (EDIT THESE)
# ==========================================================
YEAR_A = 2019
YEAR_B = 2024

# Folder structure you already use:
# data_clean/mdna_clean_2019/<firm>_2019.txt
# data_clean/mdna_clean_2024/<firm>_2024.txt
BASE = Path(".").resolve()
CLEAN_A = BASE / "data_clean" / f"mdna_clean_{YEAR_A}"
CLEAN_B = BASE / "data_clean" / f"mdna_clean_{YEAR_B}"

# OUTPUT: new dedicated folder (so it doesn’t mix with your other outputs)
OUT_DIR = BASE / "outputs" / f"firm_year_{YEAR_A}_{YEAR_B}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Put your firm tickers here (must match filenames)
# Example files:
#   ak_2019.txt and ak_2024.txt
FIRMS = [
    "ak", "ph", "aa", "tkh", "ar", "bam", "he", "br", "amg", "fu"
]

# Optional: if your filenames are NOT firm_year (e.g. "philips_2019"),
# set this to True and edit the filename templates below.
USE_CUSTOM_FILENAMES = False


# ==========================================================
# 2) HELPERS
# ==========================================================
def word_count(text: str) -> int:
    # simple robust word count
    return len(re.findall(r"\b\w+\b", text))

def load_text(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")

def cosine_sim_two_docs(text_a: str, text_b: str) -> float:
    vec = TfidfVectorizer()
    X = vec.fit_transform([text_a, text_b])
    sim = cosine_similarity(X)[0, 1]
    return float(sim)

def make_file_paths(firm: str) -> tuple[Path, Path]:
    if USE_CUSTOM_FILENAMES:
        # >>> EDIT THESE TWO LINES IF YOU NEED CUSTOM NAMES <<<
        file_a = CLEAN_A / f"{firm}.txt"
        file_b = CLEAN_B / f"{firm}.txt"
    else:
        file_a = CLEAN_A / f"{firm}_{YEAR_A}.txt"
        file_b = CLEAN_B / f"{firm}_{YEAR_B}.txt"
    return file_a, file_b


# ==========================================================
# 3) RUN: firm-year similarity (stacked)
# ==========================================================
rows = []
missing = []

for firm in FIRMS:
    file_a, file_b = make_file_paths(firm)

    if not file_a.exists() or not file_b.exists():
        missing.append({
            "firm": firm,
            "missing_year_a": str(file_a) if not file_a.exists() else "",
            "missing_year_b": str(file_b) if not file_b.exists() else ""
        })
        continue

    text_a = load_text(file_a)
    text_b = load_text(file_b)

    wc_a = word_count(text_a)
    wc_b = word_count(text_b)

    sim = cosine_sim_two_docs(text_a, text_b)

    delta_words = wc_b - wc_a
    delta_words_pct = (delta_words / wc_a) if wc_a > 0 else np.nan

    rows.append({
        "firm": firm,
        "year_a": YEAR_A,
        "year_b": YEAR_B,
        "file_a": str(file_a),
        "file_b": str(file_b),
        "words_year_a": wc_a,
        "words_year_b": wc_b,
        "delta_words": delta_words,
        "delta_words_pct": delta_words_pct,
        "cosine_similarity": sim
    })

df = pd.DataFrame(rows).sort_values("firm").reset_index(drop=True)

# Direction buckets (useful for interpretation)
df["direction"] = np.where(df["cosine_similarity"] >= 0.80, "HIGH similarity (≥0.80)",
                  np.where(df["cosine_similarity"] >= 0.70, "MED similarity (0.70–0.79)",
                  np.where(df["cosine_similarity"] >= 0.60, "LOW similarity (0.60–0.69)",
                           "VERY LOW similarity (<0.60)")))

# ==========================================================
# 4) TABLES / SUMMARIES (NO correlation step)
# ==========================================================

# 4.1 Descriptive statistics: similarity + length changes
desc = pd.DataFrame({
    "cosine_similarity": df["cosine_similarity"].describe(),
    "words_year_a": df["words_year_a"].describe(),
    "words_year_b": df["words_year_b"].describe(),
    "delta_words": df["delta_words"].describe(),
    "delta_words_pct": df["delta_words_pct"].describe()
})

# 4.2 Top / bottom similarity firms (within-firm stability)
top5_sim = df.sort_values("cosine_similarity", ascending=False).head(5).copy()
bot5_sim = df.sort_values("cosine_similarity", ascending=True).head(5).copy()

# 4.3 Biggest length changes (absolute and %)
top5_abs_len = df.reindex(df["delta_words"].abs().sort_values(ascending=False).index).head(5).copy()
top5_pct_len = df.reindex(df["delta_words_pct"].abs().sort_values(ascending=False).index).head(5).copy()

# 4.4 Category counts (easy bar chart in Excel)
direction_counts = (
    df["direction"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)
direction_counts["pct"] = direction_counts["count"] / direction_counts["count"].sum()

# 4.5 Simple “flags” for later qualitative deep dives
# (you can tune thresholds)
df["flag_large_length_change_pct"] = df["delta_words_pct"].abs() >= 0.30
df["flag_low_similarity"] = df["cosine_similarity"] <= 0.70

flags = df.loc[df["flag_large_length_change_pct"] | df["flag_low_similarity"], [
    "firm", "cosine_similarity", "words_year_a", "words_year_b", "delta_words", "delta_words_pct",
    "flag_large_length_change_pct", "flag_low_similarity"
]].copy()

# ==========================================================
# 5) EXPORT: Excel with multiple sheets + CSVs
# ==========================================================
xlsx_path = OUT_DIR / f"firm_year_{YEAR_A}_{YEAR_B}_export.xlsx"
csv_main = OUT_DIR / f"firm_year_{YEAR_A}_{YEAR_B}_main.csv"
csv_missing = OUT_DIR / f"firm_year_{YEAR_A}_{YEAR_B}_missing_files.csv"

df.to_csv(csv_main, index=False)
pd.DataFrame(missing).to_csv(csv_missing, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="firm_year_main", index=False)
    desc.to_excel(writer, sheet_name="descriptives")
    direction_counts.to_excel(writer, sheet_name="similarity_buckets", index=False)
    top5_sim.to_excel(writer, sheet_name="top5_similarity", index=False)
    bot5_sim.to_excel(writer, sheet_name="bottom5_similarity", index=False)
    top5_abs_len.to_excel(writer, sheet_name="top5_len_abs", index=False)
    top5_pct_len.to_excel(writer, sheet_name="top5_len_pct", index=False)
    flags.to_excel(writer, sheet_name="flags_for_qual", index=False)
    pd.DataFrame(missing).to_excel(writer, sheet_name="missing_files", index=False)

print("✅ Export complete!")
print("Main Excel:", xlsx_path.resolve())
print("Main CSV:", csv_main.resolve())
if missing:
    print("\n⚠️ Missing files detected. See:", csv_missing.resolve())
else:
    print("\nNo missing files. All firms processed.")
print("\nPreview (first rows):")
print(df.head(10))


✅ Export complete!
Main Excel: /Users/pnlmf036/Documents/Thesis_Textanalysis/outputs/firm_year_2019_2024/firm_year_2019_2024_export.xlsx
Main CSV: /Users/pnlmf036/Documents/Thesis_Textanalysis/outputs/firm_year_2019_2024/firm_year_2019_2024_main.csv

No missing files. All firms processed.

Preview (first rows):
  firm  year_a  year_b                                             file_a  \
0   aa    2019    2024  /Users/pnlmf036/Documents/Thesis_Textanalysis/...   
1   ak    2019    2024  /Users/pnlmf036/Documents/Thesis_Textanalysis/...   
2  amg    2019    2024  /Users/pnlmf036/Documents/Thesis_Textanalysis/...   
3   ar    2019    2024  /Users/pnlmf036/Documents/Thesis_Textanalysis/...   
4  bam    2019    2024  /Users/pnlmf036/Documents/Thesis_Textanalysis/...   
5   br    2019    2024  /Users/pnlmf036/Documents/Thesis_Textanalysis/...   
6   fu    2019    2024  /Users/pnlmf036/Documents/Thesis_Textanalysis/...   
7   he    2019    2024  /Users/pnlmf036/Documents/Thesis_Textanalysis/.